# 10 — Customer Lifetime Value

**Purpose:** Understand predicted customer value and how it aligns with segments.  
**Key question:** How much is each customer projected to spend, and who are the highest-value targets?  

**Tables used:** `marts.mart_customer_clv`, `marts.mart_cluster_output`

In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import os
PROJECT = os.environ.get('BQ_PROJECT', 'fmn-sandbox')
LOCATION = 'africa-south1'
client = bigquery.Client(project=PROJECT, location=LOCATION)

def q(sql):
    return client.query(sql).to_dataframe()

print(f'Connected to {PROJECT} ({LOCATION})')

## 1. CLV tier distribution
How many customers in each value tier?

In [ ]:
tiers = q(f"""
    SELECT clv_tier,
           COUNT(*) AS customers,
           ROUND(AVG(predicted_clv), 0) AS avg_clv,
           ROUND(SUM(predicted_clv), 0) AS total_projected_value,
           ROUND(AVG(historical_spend), 0) AS avg_historical
    FROM `{PROJECT}.marts.mart_customer_clv`
    GROUP BY clv_tier
    ORDER BY avg_clv DESC
""")

if not tiers.empty:
    display(tiers)

    fig = px.bar(tiers, x='clv_tier', y='customers', color='avg_clv',
                 color_continuous_scale='Greens',
                 title='Customers by CLV tier',
                 labels={'customers': 'Number of customers', 'avg_clv': 'Avg predicted CLV (R)'})
    fig.show()
else:
    print('No CLV data. Run predict_clv.sql first.')

## 2. CLV vs historical spend
Does predicted value align with past behaviour?

In [ ]:
if not tiers.empty:
    fig = go.Figure()
    fig.add_trace(go.Bar(name='Historical spend (R)', x=tiers['clv_tier'], y=tiers['avg_historical'],
                         marker_color='#607D8B'))
    fig.add_trace(go.Bar(name='Predicted CLV (R)', x=tiers['clv_tier'], y=tiers['avg_clv'],
                         marker_color='#2E75B6'))
    fig.update_layout(barmode='group', title='Historical spend vs predicted CLV by tier',
                      yaxis_title='R')
    fig.show()

## 3. CLV by segment
How does predicted lifetime value map to our behavioural segments?

In [ ]:
seg_clv = q(f"""
    SELECT co.segment_name,
           COUNT(*) AS customers,
           ROUND(AVG(cv.predicted_clv), 0) AS avg_clv,
           ROUND(SUM(cv.predicted_clv), 0) AS total_clv,
           ROUND(SUM(cv.predicted_clv) * 100.0 / SUM(SUM(cv.predicted_clv)) OVER(), 1) AS pct_of_total_value
    FROM `{PROJECT}.marts.mart_cluster_output` co
    JOIN `{PROJECT}.marts.mart_customer_clv` cv ON co.UNIQUE_ID = cv.UNIQUE_ID
    GROUP BY co.segment_name
    ORDER BY avg_clv DESC
""")

if not seg_clv.empty:
    display(seg_clv)

    fig = go.Figure()
    fig.add_trace(go.Bar(name='% of customers', x=seg_clv['segment_name'],
                         y=seg_clv['customers'] * 100.0 / seg_clv['customers'].sum(),
                         marker_color='#607D8B'))
    fig.add_trace(go.Bar(name='% of projected value', x=seg_clv['segment_name'],
                         y=seg_clv['pct_of_total_value'], marker_color='#2E75B6'))
    fig.update_layout(barmode='group', title='Customer share vs value share by segment',
                      yaxis_title='%')
    fig.show()

    champ = seg_clv[seg_clv['segment_name'] == 'Champions'].iloc[0]
    print(f'\nChampions represent {champ["pct_of_total_value"]}% of total projected value')

## 4. CLV by demographics
Which age and income groups have the highest predicted value?

In [ ]:
demo_clv = q(f"""
    SELECT age_group,
           COUNT(*) AS customers,
           ROUND(AVG(predicted_clv), 0) AS avg_clv
    FROM `{PROJECT}.marts.mart_customer_clv`
    WHERE age_group IS NOT NULL
    GROUP BY age_group ORDER BY age_group
""")

if not demo_clv.empty:
    fig = px.bar(demo_clv, x='age_group', y='avg_clv', color='customers',
                 color_continuous_scale='Blues',
                 title='Average predicted CLV by age group',
                 labels={'avg_clv': 'Avg CLV (R)'})
    fig.show()

In [ ]:
income_clv = q(f"""
    SELECT income_group,
           COUNT(*) AS customers,
           ROUND(AVG(predicted_clv), 0) AS avg_clv
    FROM `{PROJECT}.marts.mart_customer_clv`
    WHERE income_group IS NOT NULL
    GROUP BY income_group ORDER BY avg_clv DESC
""")

if not income_clv.empty:
    fig = px.bar(income_clv, x='income_group', y='avg_clv', color='customers',
                 color_continuous_scale='Oranges',
                 title='Average predicted CLV by income group',
                 labels={'avg_clv': 'Avg CLV (R)'})
    fig.show()

## 5. High value at-risk customers
Customers with high CLV but also high churn risk — these are the priority saves.

In [ ]:
at_risk_value = q(f"""
    SELECT cv.clv_tier, cr.churn_risk_level,
           COUNT(*) AS customers,
           ROUND(SUM(cv.predicted_clv), 0) AS projected_value_at_risk
    FROM `{PROJECT}.marts.mart_customer_clv` cv
    JOIN `{PROJECT}.marts.mart_churn_risk` cr ON cv.UNIQUE_ID = cr.UNIQUE_ID
    WHERE cv.clv_tier IN ('Platinum', 'Gold')
      AND cr.churn_risk_level IN ('Critical', 'High')
    GROUP BY cv.clv_tier, cr.churn_risk_level
    ORDER BY projected_value_at_risk DESC
""")

if not at_risk_value.empty:
    display(at_risk_value)
    total_at_risk = at_risk_value['projected_value_at_risk'].sum()
    total_custs = at_risk_value['customers'].sum()
    print(f'\n{total_custs:,} high-value customers at risk of churning')
    print(f'Projected value at risk: R{total_at_risk:,.0f}')
    print(f'A 10% save rate recovers R{total_at_risk * 0.1:,.0f}')
else:
    print('No high-value customers at risk. Good news.')